# ML System Design: Recommendation System

Applying the universal design template to build a recommendation system end-to-end.

This notebook covers:
1. System design walkthrough (all 8 stages)
2. Collaborative filtering approaches
3. Content-based filtering
4. Modern neural approaches (two-tower, cross-network)
5. Full retrieval → ranking → re-ranking pipeline
6. Implementation: Matrix Factorization with ALS from scratch

---

## Stage 1 — Problem Clarification

**Prompt**: "Design a recommendation system for a video streaming platform."

### Clarifying Questions & Assumed Answers

| Question | Answer |
|----------|--------|
| What are we recommending? | Videos (movies, TV episodes, short clips) |
| Where do recs appear? | Homepage, "More Like This", post-play |
| Scale? | 200M monthly active users, 500K videos in catalog |
| Latency? | < 200ms for homepage load |
| What signals do we have? | Watch history, ratings, search queries, browse behavior |
| Cold start? | ~10% of users are new (< 5 interactions), ~1K new videos/day |
| Current solution? | Popularity-based + editorial curated lists |
| Objective? | Increase watch time and reduce churn |

---
## Stage 2 — Metrics

### Offline Metrics

| Metric | What it Measures | Notes |
|--------|-----------------|-------|
| **Recall@K** | Of items user engaged with, how many were in top-K? | K=20 for retrieval, K=10 for ranking |
| **NDCG@K** | Ranking quality with graded relevance | Weight by engagement level: skip < partial_watch < full_watch < rate |
| **Hit Rate@K** | Did at least one relevant item appear in top-K? | Simpler than NDCG, good for retrieval |
| **MAP** | Mean average precision | Good for binary relevance |

### Online Metrics

| Metric | Priority |
|--------|----------|
| **Watch time** | Primary (minutes watched per session) |
| **Completion rate** | How often users finish a video |
| **D7/D30 retention** | Long-term health |
| **Exploration rate** | Are users discovering new genres? |

### Guardrail Metrics

- **Diversity**: catalog coverage (% of catalog recommended), genre entropy
- **Freshness**: avg age of recommended items
- **Fairness**: exposure across content creators

### Key insight
Optimizing purely for watch time can lead to clickbait and filter bubbles. Guardrail metrics prevent degenerate solutions.

---
## Stage 3 — Data

### Interaction Data

| Signal | Type | Volume |
|--------|------|--------|
| Video watches (with duration) | Implicit, strong | ~1B/day |
| Explicit ratings (1-5 stars) | Explicit, sparse | ~10M/day |
| Search queries | Implicit | ~500M/day |
| Clicks (browse, thumbnail) | Implicit, weak | ~5B/day |
| Skips / back-button | Implicit, negative | ~2B/day |

### Side Information

- **User**: country, language, subscription tier, device, age group
- **Video**: title, genre, cast, director, duration, release date, description embedding, thumbnail embedding
- **Context**: time of day, day of week, recently watched

### Label Construction

For ranking model, define engagement as:
```
label = 1.0 if watched > 80% of video
        0.5 if watched 30-80%
        0.0 if watched < 30% or skipped
```

### Training-Serving Consistency

Log features at serving time ("logged features") and join with outcomes to create training data. This avoids training-serving skew.

---
## Collaborative Filtering: Concepts

### User-User Collaborative Filtering

**Idea**: Users who agreed in the past will agree in the future.

1. Find users similar to target user (cosine similarity on interaction vectors)
2. Recommend items liked by similar users that target hasn't seen

**Pros**: Simple, no item features needed  
**Cons**: Doesn't scale (computing pairwise user similarity is O(n^2)), cold start for new users

### Item-Item Collaborative Filtering

**Idea**: Items consumed by similar sets of users are similar.

1. Precompute item-item similarity matrix
2. For a user, look at items they liked, find similar items

**Pros**: More stable than user-user (item tastes change slower), precomputable  
**Cons**: Still struggles with cold start for new items

### Matrix Factorization

**Idea**: Decompose the user-item interaction matrix R (m users x n items) into two low-rank matrices:

$$R \approx U \cdot V^T$$

where:
- $U \in \mathbb{R}^{m \times k}$ — user embeddings
- $V \in \mathbb{R}^{n \times k}$ — item embeddings
- $k$ — latent dimension (typically 50-200)

**Prediction**: $\hat{r}_{ui} = u_i^T v_j + b_u + b_i + \mu$

**Training**: Minimize squared error on observed ratings with regularization:

$$\min_{U, V} \sum_{(u,i) \in \text{observed}} (r_{ui} - u_u^T v_i)^2 + \lambda(\|U\|^2 + \|V\|^2)$$

**Methods**:
- **SGD**: update U, V alternately with gradient steps
- **ALS** (Alternating Least Squares): fix U, solve for V in closed form, then fix V, solve for U. Parallelizable!

---
## Implementation: Matrix Factorization with ALS from Scratch

### ALS Algorithm

When we fix U, the optimization over V decomposes into independent least-squares problems per item. For item $i$:

$$v_i = (U_{I_i}^T U_{I_i} + \lambda I)^{-1} U_{I_i}^T r_{I_i, i}$$

where $I_i$ is the set of users who rated item $i$, $U_{I_i}$ is the submatrix of U for those users, and $r_{I_i, i}$ is the vector of their ratings.

Similarly when we fix V, we solve for each user's embedding.

In [ ]:
import numpy as np
from typing import Tuple


class ALSMatrixFactorization:
    """
    Alternating Least Squares for explicit-feedback matrix factorization.
    
    Given a sparse user-item rating matrix R (m x n), factorize into:
        R ≈ U @ V.T
    where U is (m x k) and V is (n x k).
    
    ALS alternates between:
        1. Fix V, solve for U (closed-form least squares per user)
        2. Fix U, solve for V (closed-form least squares per item)
    """
    
    def __init__(self, n_factors: int = 20, reg: float = 0.1, n_iterations: int = 20,
                 seed: int = 42):
        self.n_factors = n_factors
        self.reg = reg
        self.n_iterations = n_iterations
        self.seed = seed
        self.U = None  # user factors (m x k)
        self.V = None  # item factors (n x k)
    
    def fit(self, R: np.ndarray, verbose: bool = True) -> 'ALSMatrixFactorization':
        """
        Fit the model to a (possibly sparse) rating matrix R.
        R[u, i] = rating if observed, 0 if unobserved.
        We only fit on observed (nonzero) entries.
        """
        np.random.seed(self.seed)
        m, n = R.shape
        k = self.n_factors
        
        # Initialize factors with small random values
        self.U = np.random.randn(m, k) * 0.1
        self.V = np.random.randn(n, k) * 0.1
        
        # Mask of observed entries
        observed = (R != 0)
        
        # Regularization matrix
        reg_I = self.reg * np.eye(k)
        
        for iteration in range(self.n_iterations):
            # --- Step 1: Fix V, solve for each user u ---
            for u in range(m):
                # Indices of items rated by user u
                rated_items = np.where(observed[u, :])[0]
                if len(rated_items) == 0:
                    continue
                
                # V_sub: (num_rated x k)
                V_sub = self.V[rated_items, :]
                # r_sub: ratings by user u for those items
                r_sub = R[u, rated_items]
                
                # Closed-form: u_u = (V_sub^T V_sub + lambda*I)^{-1} V_sub^T r_sub
                A = V_sub.T @ V_sub + reg_I
                b = V_sub.T @ r_sub
                self.U[u, :] = np.linalg.solve(A, b)
            
            # --- Step 2: Fix U, solve for each item i ---
            for i in range(n):
                # Indices of users who rated item i
                rated_users = np.where(observed[:, i])[0]
                if len(rated_users) == 0:
                    continue
                
                U_sub = self.U[rated_users, :]
                r_sub = R[rated_users, i]
                
                A = U_sub.T @ U_sub + reg_I
                b = U_sub.T @ r_sub
                self.V[i, :] = np.linalg.solve(A, b)
            
            # Compute loss on observed entries
            if verbose:
                loss = self._compute_loss(R, observed)
                print(f"Iteration {iteration + 1:3d} | Loss: {loss:.4f}")
        
        return self
    
    def _compute_loss(self, R: np.ndarray, observed: np.ndarray) -> float:
        """Compute regularized MSE on observed entries."""
        R_pred = self.U @ self.V.T
        diff = (R - R_pred) * observed  # zero out unobserved entries
        mse = np.sum(diff ** 2) / np.sum(observed)
        reg_term = self.reg * (np.sum(self.U ** 2) + np.sum(self.V ** 2))
        return mse + reg_term
    
    def predict(self, user_idx: int, item_idx: int) -> float:
        """Predict rating for a single (user, item) pair."""
        return self.U[user_idx] @ self.V[item_idx]
    
    def recommend(self, user_idx: int, R: np.ndarray, top_k: int = 10) -> np.ndarray:
        """Recommend top-k items for a user (excluding already-rated items)."""
        scores = self.U[user_idx] @ self.V.T
        # Mask out already-rated items
        already_rated = np.where(R[user_idx, :] != 0)[0]
        scores[already_rated] = -np.inf
        return np.argsort(scores)[::-1][:top_k]

In [ ]:
# ============================================================
# Generate synthetic rating data and test ALS
# ============================================================

np.random.seed(42)

# Ground truth: there are 5 latent "taste" dimensions
n_users, n_items, true_k = 200, 100, 5
U_true = np.random.randn(n_users, true_k)
V_true = np.random.randn(n_items, true_k)
R_true = U_true @ V_true.T

# Clip to [1, 5] rating range
R_true = np.clip(R_true, 1, 5)

# Observe only 20% of entries (simulate sparsity)
mask = np.random.rand(n_users, n_items) < 0.20
R_observed = R_true * mask  # unobserved entries are 0

sparsity = 1 - np.sum(mask) / (n_users * n_items)
print(f"Rating matrix: {n_users} users x {n_items} items")
print(f"Observed entries: {np.sum(mask):,} ({100*(1-sparsity):.1f}%)")
print(f"Sparsity: {sparsity:.1%}")

In [ ]:
# Train ALS
als = ALSMatrixFactorization(n_factors=10, reg=0.1, n_iterations=20)
als.fit(R_observed)

In [ ]:
# Evaluate: compute RMSE on held-out entries
held_out = (~mask) & (R_true > 0)  # unobserved entries with true ratings
R_pred = als.U @ als.V.T

# RMSE on held-out
errors = (R_true[held_out] - R_pred[held_out])
rmse = np.sqrt(np.mean(errors ** 2))
print(f"Held-out RMSE: {rmse:.4f}")

# Compare with baseline: predict global mean
global_mean = np.mean(R_observed[R_observed > 0])
baseline_rmse = np.sqrt(np.mean((R_true[held_out] - global_mean) ** 2))
print(f"Global mean baseline RMSE: {baseline_rmse:.4f}")
print(f"Improvement: {(baseline_rmse - rmse) / baseline_rmse:.1%}")

In [ ]:
# Recommend top-10 items for user 0
top_items = als.recommend(user_idx=0, R=R_observed, top_k=10)
print(f"Top 10 recommended items for user 0: {top_items}")
print(f"Predicted scores: {[f'{als.predict(0, i):.2f}' for i in top_items]}")

---
## Evaluation Metrics for Recommendations

In [ ]:
def recall_at_k(recommended: np.ndarray, relevant: np.ndarray, k: int) -> float:
    """What fraction of relevant items appear in the top-k recommendations?"""
    rec_at_k = set(recommended[:k])
    rel = set(relevant)
    if len(rel) == 0:
        return 0.0
    return len(rec_at_k & rel) / len(rel)


def ndcg_at_k(recommended: np.ndarray, relevance_scores: dict, k: int) -> float:
    """
    Normalized Discounted Cumulative Gain.
    
    relevance_scores: dict mapping item_id -> relevance (e.g., rating)
    """
    # DCG for the recommended list
    dcg = 0.0
    for i, item in enumerate(recommended[:k]):
        rel = relevance_scores.get(item, 0)
        dcg += (2 ** rel - 1) / np.log2(i + 2)  # i+2 because position is 1-indexed
    
    # Ideal DCG: sort all relevant items by relevance, compute DCG
    ideal_rels = sorted(relevance_scores.values(), reverse=True)[:k]
    idcg = sum((2 ** rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_rels))
    
    if idcg == 0:
        return 0.0
    return dcg / idcg


def coverage(all_recommendations: list, n_items: int) -> float:
    """What fraction of the catalog appears in any recommendation?"""
    all_recommended = set()
    for recs in all_recommendations:
        all_recommended.update(recs)
    return len(all_recommended) / n_items


# Evaluate our ALS model
k = 10
recalls, ndcgs = [], []
all_recs = []

for user in range(min(50, n_users)):  # evaluate on first 50 users
    # "Relevant" items: items with true rating > 3.5 that were not observed
    unobserved = np.where(~mask[user])[0]
    relevant = unobserved[R_true[user, unobserved] > 3.5]
    
    if len(relevant) == 0:
        continue
    
    # Get recommendations
    recs = als.recommend(user, R_observed, top_k=k)
    all_recs.append(recs)
    
    recalls.append(recall_at_k(recs, relevant, k))
    rel_scores = {i: R_true[user, i] for i in relevant}
    ndcgs.append(ndcg_at_k(recs, rel_scores, k))

print(f"Mean Recall@{k}: {np.mean(recalls):.4f}")
print(f"Mean NDCG@{k}: {np.mean(ndcgs):.4f}")
print(f"Catalog Coverage: {coverage(all_recs, n_items):.2%}")

---
## Content-Based Filtering

**Idea**: Recommend items similar to what the user liked, based on item features (not other users' behavior).

### Approach

1. Represent each item as a feature vector (genre, cast, description embedding, etc.)
2. Build a user profile as a weighted average of item features they liked
3. Score new items by similarity (cosine) between user profile and item feature vector

### Pros
- No cold start for new items (if they have features)
- Explainable ("Because you liked action movies")
- No need for other users' data

### Cons
- Limited novelty / serendipity (recommends similar items)
- Feature engineering burden
- Cold start for new users (no history to build profile)

### In Practice
Content-based is often combined with collaborative filtering (hybrid approach). The content features serve as side information in the collaborative model.

---
## Modern Neural Approaches

### Two-Tower Model (Retrieval Stage)

```
User Tower:                  Item Tower:
  user_id → embedding          item_id → embedding
  user_features → MLP          item_features → MLP
        ↓                           ↓
  user_embedding (d=128)       item_embedding (d=128)
        \                         /
         → dot product / cosine →  score
```

**Key properties**:
- User and item embeddings are computed independently
- At serving time, precompute all item embeddings, use ANN (FAISS, ScaNN) for fast retrieval
- Scales to millions of items with sub-millisecond retrieval
- Training: in-batch negatives (other items in the same batch serve as negatives)

**Limitations**: No cross-features between user and item (by design, for efficiency)

### Cross-Network / Deep & Cross (Ranking Stage)

After retrieval narrows candidates to ~1000, the ranking model can be more expressive:

```
Input: [user_features, item_features, cross_features, context]
  → Cross Network (explicit feature interactions)
  → Deep Network (implicit feature interactions)
  → Combine → sigmoid → P(engagement)
```

The cross network explicitly models feature interactions up to order L (L cross layers), complementing the DNN.

### Full Pipeline

```
                    500K items
                        ↓
Candidate Generation  [Two-Tower + ANN → 1000 candidates]
                        ↓
Ranking               [DCN/DeepFM scoring → top 50]
                        ↓
Re-ranking            [Diversity, freshness, business rules → final 20]
```

### Re-ranking objectives
- **Diversity**: MMR (Maximal Marginal Relevance) — balance relevance and diversity
- **Freshness**: boost recently added content
- **Business rules**: promoted content, contractual obligations, content policies
- **Deduplication**: avoid showing too many items from same creator/genre

---
## Cold Start Problem

One of the most asked follow-up questions in interviews.

### Cold Start for New Users

| Strategy | How |
|----------|-----|
| **Popularity-based** | Show globally popular items as default |
| **Onboarding** | Ask user to pick genres/titles they like |
| **Contextual** | Use available signals: country, device, time, referral source |
| **Bandit exploration** | Use Thompson sampling / epsilon-greedy to explore user preferences |
| **Transfer** | If user has account on related platform, transfer preferences |

### Cold Start for New Items

| Strategy | How |
|----------|-----|
| **Content features** | Use item metadata (genre, description embedding) for content-based recs |
| **Exploration** | Inject new items into a small % of recommendations to collect signal |
| **Similar items** | Find existing items with similar metadata, use their engagement patterns |
| **Creator-based** | Items from popular creators inherit initial priors |

---
## Stage 4-8 — Remaining Design Stages

### Feature Engineering (Stage 4)

| Feature Group | Examples |
|--------------|----------|
| User static | country, language, subscription tier, account age |
| User behavioral | avg watch time/session, genre distribution, time-of-day patterns |
| Item static | genre, duration, release year, cast popularity |
| Item behavioral | global CTR, completion rate, avg rating |
| User-Item cross | user genre preference x item genre, past interactions with creator |
| Context | hour, day_of_week, device, session length so far |
| Sequence | last 20 watched items (as embedding sequence) |

### Training (Stage 6)

- Two-tower: daily retraining, sample negatives from in-batch or hard negatives
- Ranker: daily retraining, temporal split, last 14 days of data
- Multi-task: predict P(click), P(complete), P(rate), P(share) with shared layers

### Serving (Stage 7)

- Item embeddings refreshed daily, indexed in FAISS/ScaNN
- User embedding computed on-the-fly (or cached with TTL)
- Ranking model served via TF Serving / Triton
- Total latency budget: retrieval ~20ms + ranking ~80ms + re-ranking ~10ms + overhead ~90ms = ~200ms
- A/B test: user-level randomization, 2-week minimum, track watch time + retention

### Monitoring (Stage 8)

- Daily: NDCG on logged data, recommendation freshness, coverage
- Weekly: calibration check (does predicted P(click) match actual CTR?)
- Alert: >3% drop in watch time per session, >10% drop in coverage
- Feedback loop: popularity bias → inject exploration, track long-tail exposure

---
## Key Interview Talking Points

1. **Always start with the pipeline**: retrieval → ranking → re-ranking. This shows you understand scale.
2. **Cold start is almost always asked**. Have solutions ready for both user and item cold start.
3. **Offline-online metric gap**: NDCG improvement offline does not guarantee watch time improvement online. Discuss the gap.
4. **Position bias**: items shown higher get more clicks regardless of relevance. Discuss inverse propensity weighting or position-aware models.
5. **Diversity vs relevance tradeoff**: pure relevance optimization creates filter bubbles. Discuss MMR, exploration, diversity constraints.
6. **Two-tower is the workhorse**: for large-scale retrieval, this is the industry standard. Know it well.